# Quickstart


## Overview

* [Your first topology](#first)
* [The example data](#data)
* [The code](#code)
* [Start the stream processing](#start)

---
<a id="first"></a>
## Your first topology

A processing pipeline is called *topology* in Kafi Streams - as already in Kafka Streams.

A Kafi Streams topology is a directed acyclic graph of operators starting with arbitrary many *sources* and ending with arbitrary many *sinks* (typically corresponding to Kafka topics).

Here is a concrete example, displayed similar to the way we are used to look at Kafka Streams topologies-(see https://zz85.github.io/kafka-streams-viz/):

```mermaid
graph TD
1c1da500-465b-476b-8c16-89a987640b49[source_clicks] --> 8eec074e-81a4-47cd-8616-c759746c0171[map_op]
5b497318-5974-47c0-ae95-e6188dd5b4c1[source_customers] --> 86f7c0da-6ed1-49d0-a9ab-d4b2a0d3612c[map_op]
8eec074e-81a4-47cd-8616-c759746c0171[map_op] --> 91e83a73-5f38-4f5c-b0de-e76239442040[filter_op]
91e83a73-5f38-4f5c-b0de-e76239442040[filter_op] --> 93b9a183-e4fb-4b53-8109-3f7fb2176b1b[distinct_op]
00a0838d-dcd1-481e-aaf0-eef658907875[join_op] --> f4895ed0-294a-4cea-a33c-e0e334c46670[sink_joined]
93b9a183-e4fb-4b53-8109-3f7fb2176b1b[distinct_op] --> 00a0838d-dcd1-481e-aaf0-eef658907875[join_op]
1e14b3dd-2ea3-4dce-9ed8-912ba3e81a8b[distinct_op] --> 00a0838d-dcd1-481e-aaf0-eef658907875[join_op]
86f7c0da-6ed1-49d0-a9ab-d4b2a0d3612c[map_op] --> 1e14b3dd-2ea3-4dce-9ed8-912ba3e81a8b[distinct_op]
```
There are two sources (`source_clicks` and `source_customers`) at the top.

* `source_clicks`: the clicks go through a `map()` operator and then a `filter()`. The `distinct()` operator cleans up duplicates.
* `source_customers`: the customers just go through a `map()`. The `distinct()` operator cleans up duplicates.

Then, both sides are joined by the `join()` operator and end up in the sink (`sink_joined`).

> Typically, both the sources and the sinks are Kafka topics, but it is also already possible to just call an arbitrary function for each record landing in the sink, e.g. for direct ingestion into a target database. This is described in [Lifecycle (Streams.sink_fun)](lifecycle.ipynb).

---
<a id="data"></a>
## The example data

Before we can begin to turn to the code, we need some example data.

We assume that `clicks` is a source of Kafka messages like this:

In [ ]:
{
    "key": None,
    "value": {"customer_id": "4711",
              "view_time": 200,
              "ts": 1609457200000},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and that `customers` is a source of Kafka messages like this:

In [ ]:
{
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` (on the `customer_id`) to enrich the fields from the `clicks` with the `name` of the customer from the `customers`.

An example output message looks like this (only the `value` field of the Kafka message is relevant here):


In [ ]:
{"value": {"customer_id": "4711",   # customer ID (equi join key)
           "view_time": 200,        # view time (from clicks)
           "ts": 1609457200000,     # timestamp (from clicks) 
           "name": "Sallyann Jupp"} # name (from customers)
}

The data generators for `clicks` and `customers` are here: [generators.py](generators.py).

They do need a library (`faker`) that we install as follows:

In [ ]:
!pip install -r requirements.txt

---
<a id="code"></a>
## The code

Now we can start coding the topology.

In Kafi Streams, you specify topologies using a Kafka Streams DSL-inspired fluent API:


In [2]:
# 1. Boilerplate

import sys
sys.path.insert(1, "../..")

from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.INFO)

# 2. Connect to Kafka

from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

# 3. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"

## a) Clicks

click_tn = (
    Streams.source(c, click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Streams.source(c, customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

sink_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(c, sink_str)
)

# 4. Build the Topology

tn = Streams.build(sink_tn)


Let's go through the code in baby steps.

### 1. Boilerplate

We start with some boilerplate. We import `Streams`, import and set up the generators and the logging.

`Streams` is a subclass of the `TopologyNode` class:
* `TopologyNode`: pydbsp-based stream processing; no dependency at all to Kafka
* `Streams`: subclass of `TopologyNode`; Kafi/Kafka wrapper around `TopologyNode`

### 2. Connect to Kafka

We connect to Kafka. In the example, we connect to a locally installed Kafka cluster.

### 3. Specify the Topology

This is the most interested step: We specify our first Kafi Streams topology. It consists of three parts.

#### a) Clicks

The first part of the topology is about the clicks:
1. We define the source with `Streams.source()`. As for the arguments, `c` stands for the local Kafka cluster and `click_source_str` for the topic name. As you can see, contrary to Kafka Streams, all sources and sinks can be on any Kafka cluster.
2. We use the `map` operator to select two fields from the value of the incoming click events (`customer_id`, `view_time` and `ts` from the `value`).
3. We employ the `filter` operator to filter out those incoming click events where `view_time` is greater than `20`.

#### b) Customers

The second part of the topology is about the customers:
1. We define the source.
2. We select two fields from the value of the incoming customer messages (`id` and `name`).

### c) Join and Sink

The third and last part of the topology joins the clicks and customers using an equi join (`join`) and sinks the result.

The arguments of the `join()` operator are:
1. Right side of the join (here: `customer_tn`)
2. Left side join key (here: the `customer_id` field of the clicks)
3. Right side join key (here: the `id` field of the customers)
4. Projection function. Here: `customer_id`, `view_time` and `ts` from the clicks, and `name` from the customers.

At the end of the topology is the sink specification (cluster `c` and topic name `sink_str`).

### 4. Building the Topology

Before you can use a Kafi Streams topology, it needs to be "built". Under the covers, this creates a pydbsp "circuit" that is eventually used for the processing.


---
<a id="start"></a>
## Start the processing

Now we are ready to rumble. Let's reset the topology, (re-)create the sources and the sink, and start the processing:

In [3]:
tn.reset()

c.recreate(click_source_str)
c.recreate(customer_source_str)
c.recreate(sink_str)

stop = Streams.start_streams(tn, progress=True)


INFO:kafi.streams.streams:Starting Streams...


(['clicks'], 'streams_1788179204313')
(['customers'], 'streams_1788179204313')


After this step, Kafi Streams is still idling around as there is no data coming.

So let's produce some data to the two input topics (10.000 messages to `clicks` and 10.000 to `customers`) and see what happens:

In [4]:
from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_pr = c.producer(click_source_str)
customer_pr = c.producer(customer_source_str)

for i in range(100):
    click_m_dict_list = click_generator.generate(100)
    click_pr.produce_list(click_m_dict_list)

    customer_m_dict_list = customer_generator.generate(100)
    customer_pr.produce_list(customer_m_dict_list)

click_pr.close()
customer_pr.close()


Uptime: 0.083s, State size: 852.86 KB, Source offsets: {'clicks': {0: 4000}, 'customers': {0: 4000}}, Sink outputs: 3569

'customers'

Uptime: 10.018s, State size: 942.86 KB, Source offsets: {'clicks': {0: 10000}, 'customers': {0: 10000}}, Sink outputs: 89744

When we see that Kafi Streams has completely read the source topics (=the source offsets have both reached `10000`), you can produce more data at will.

Or stop the Kafi Streams processing thread like so (`Streams.threads()` should confirm that there are no remaining Streams threads after stopping):

In [ ]:
stop()
Streams.threads()

Let's check whether the sink topic has the same number of messages as displayed by the Kafi Streams progress indicator (`Sink outputs`):

In [ ]:
c.l(sink_str)

And, last but not least, let's have a look at the last ten messages in the sink topic:

In [ ]:
c.cat(sink_str, last_n=10)

So that's it. Congratulations - that was your first Streams topology in action :-)
